# TIFF metadata calibration -> um/px per image

Recover the real um/px from each TIFF's embedded metadata - **Olympus SIS** (`sis_metadata`,
`pixelsizex` in metres), **OME-XML** (`PhysicalSizeX`), **ImageJ**, or resolution tags - and
write `scale_table.csv`. Phase 1 looks up um/px by filename.

> Sanity-check against a printed scale bar on one image before trusting it for the cohort.

In [ ]:
# pip install tifffile numpy pandas
import numpy as np
import pandas as pd
import tifffile
import xml.etree.ElementTree as ET
from pathlib import Path
print('ready')

In [ ]:
# ======================= CONFIG - EDIT =======================
DATA_DIR = Path('path/to/your/images')   # <- EDIT THIS: the folder with your .tif images
OUT_DIR  = Path('./outputs_calib'); OUT_DIR.mkdir(exist_ok=True)
IMG_EXTS = {'.tif', '.tiff'}

# de-duplicate by filename: controls/GtElk is a byte-identical COPY of GtElk/
_seen = set(); image_paths = []
for p in sorted(DATA_DIR.rglob('*')):
    if p.suffix.lower() in IMG_EXTS and p.name not in _seen:
        _seen.add(p.name); image_paths.append(p)
print(f'{len(image_paths)} unique images (deduplicated) under {DATA_DIR}')

## Inspect headers, then build the table
The first cell shows what calibration is stored. The second recovers um/px for every image.

In [ ]:
# Inspect what's actually in the TIFF headers for a few images
for p in image_paths[:5]:
    with tifffile.TiffFile(p) as tf:
        pg = tf.pages[0]
        print('===', p.name, '| imagej:', tf.is_imagej, '| ome:', tf.is_ome)
        for tag in ('XResolution', 'YResolution', 'ResolutionUnit'):
            tg = pg.tags.get(tag)
            print('    ', tag, '=', (tg.value if tg else None))
        try:
            print('     sis_metadata:', tf.sis_metadata)   # Olympus SIS -> pixelsizex in METERS
        except Exception as e:
            print('     sis_metadata: none', e)
        if tf.is_imagej:
            print('     imagej_metadata:', tf.imagej_metadata)
        desc = pg.tags.get('ImageDescription')
        if desc:
            print('     ImageDescription:', str(desc.value)[:300])

In [ ]:
def _to_um(val, unit):
    u = (unit or '').lower()
    if u in ('um', 'micron', 'microns', 'micrometer', 'micrometre'): return val
    if u in ('nm', 'nanometer'): return val / 1000.0
    if u in ('mm', 'millimeter'): return val * 1000.0
    if u in ('m', 'meter'): return val * 1e6
    return val   # assume microns if unit is absent

def um_per_px_from_metadata(path):
    try:
        with tifffile.TiffFile(path) as tf:
            pg = tf.pages[0]
            xres = pg.tags.get('XResolution')
            # 0) Olympus SIS (sis_metadata): pixelsizex is in METERS
            try:
                sis = tf.sis_metadata
                if sis and sis.get('pixelsizex'):
                    return float(sis['pixelsizex']) * 1e6
            except Exception:
                pass
            # 1) OME-XML PhysicalSizeX
            if tf.is_ome and tf.ome_metadata:
                try:
                    for el in ET.fromstring(tf.ome_metadata).iter():
                        if el.tag.endswith('Pixels') and el.get('PhysicalSizeX'):
                            return _to_um(float(el.get('PhysicalSizeX')),
                                          el.get('PhysicalSizeXUnit', 'um'))
                except Exception:
                    pass
            # 2) ImageJ: unit + XResolution (stored as pixels per unit)
            ij = tf.imagej_metadata or {}
            if ij.get('unit') and xres is not None:
                num, den = xres.value
                if num:
                    return _to_um(den / num, ij.get('unit'))
            # 3) plain TIFF resolution in cm/inch (often print DPI - VERIFY before trusting)
            unit = pg.tags.get('ResolutionUnit')
            if xres is not None and unit is not None and unit.value in (2, 3):
                num, den = xres.value
                if num:
                    per_unit_um = 1e4 if unit.value == 3 else 25400.0   # cm or inch -> um
                    return per_unit_um / (num / den)
    except Exception:
        pass
    return np.nan

mt = pd.DataFrame([dict(image=p.name, um_per_px=um_per_px_from_metadata(p)) for p in image_paths])
mt.to_csv(OUT_DIR / 'scale_table.csv', index=False)
got = int(mt['um_per_px'].notna().sum())
print(f'Scale recovered for {got}/{len(mt)} images -> wrote scale_table.csv')
print(mt['um_per_px'].describe())
missing = mt[mt['um_per_px'].isna()]['image'].tolist()
if missing:
    print(f'{len(missing)} images have NO embedded scale (they will fall back to pixels):')
    print(missing[:20])
mt.head()

## Use it in Phase 1
Phases 1 and 2 already load this table:

```python
_SCALE = pd.read_csv('outputs_calib/scale_table.csv').set_index('image')['um_per_px'].to_dict()
def get_um_per_px(path, meta):
    return _SCALE.get(path.name, np.nan)
```